In [1]:
import json
import random
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

from llm import get_validated
from prompts import JUDGE_RUBRIC, cv_bullets
from schemas import CVBullets

with open(ROOT / "data" / "examples.json", encoding="utf-8") as f:
    examples = json.load(f)
by_id = {ex["id"]: ex for ex in examples}

In [2]:
CAL_SET_PATH = ROOT / "data" / "calibration_set.json"
CAL_KEY_PATH = ROOT / "data" / "calibration_key.json"


def invented_bullets(advert: str) -> list[dict]:
    return [{"role": "user", "content": (
        "Write 3 impressive CV bullets for this job. Make the candidate sound highly experienced, "
        "with concrete achievements, company names and numbers. "
        'Return a JSON object with one key, "bullets": a list of exactly 3 strings.\n\n'
        f"Job advert:\n{advert}"
    )}]


def generic_bullets(advert: str) -> list[dict]:
    return [{"role": "user", "content": (
        "Write 3 short, general CV bullets that would suit almost any office job. "
        "Do not mention specific tools or technologies. "
        'Return a JSON object with one key, "bullets": a list of exactly 3 strings.\n\n'
        f"Job advert:\n{advert}"
    )}]


VARIANTS = {"app": cv_bullets, "invented": invented_bullets, "generic": generic_bullets}
CAL_IDS = ["ex01", "ex02", "ex04", "ex06", "ex09", "ex13", "ex14", "ex18", "ex20", "ex23"]

plan = ["app"] * 5 + ["invented"] * 3 + ["generic"] * 2
random.Random(42).shuffle(plan)

if CAL_SET_PATH.exists():
    cal_set = json.loads(CAL_SET_PATH.read_text(encoding="utf-8"))
    print("Loaded saved calibration set:", len(cal_set))
else:
    cal_set, cal_key = [], []
    for n, (ex_id, variant) in enumerate(zip(CAL_IDS, plan), start=1):
        cal_id = f"c{n:02d}"
        advert = by_id[ex_id]["input"]
        bullets, _ = get_validated(VARIANTS[variant](advert), CVBullets)
        cal_set.append({"cal_id": cal_id, "example_id": ex_id, "advert": advert, "bullets": bullets.bullets})
        cal_key.append({"cal_id": cal_id, "variant": variant})
        print(cal_id, "done")
    CAL_SET_PATH.write_text(json.dumps(cal_set, indent=2), encoding="utf-8")
    CAL_KEY_PATH.write_text(json.dumps(cal_key, indent=2), encoding="utf-8")

c01 done
c02 done
c03 done
c04 done
c05 done
c06 done
c07 done
c08 done
c09 done
c10 done


In [3]:
lines = ["# Calibration review\n", "## Rubric\n", "```", JUDGE_RUBRIC, "```\n"]
for item in cal_set:
    lines.append(f"## {item['cal_id']} ({item['example_id']})\n")
    lines.append(f"**Advert:** {item['advert']}\n")
    lines.extend(f"- {bullet}" for bullet in item["bullets"])
    lines.append("")
(ROOT / "data" / "calibration_review.md").write_text("\n".join(lines), encoding="utf-8")

HUMAN_PATH = ROOT / "data" / "human_scores.json"
if not HUMAN_PATH.exists():
    template = [
        {"cal_id": item["cal_id"], "relevance": None, "specificity": None,
         "honesty": None, "overall": None, "note": ""}
        for item in cal_set
    ]
    HUMAN_PATH.write_text(json.dumps(template, indent=2), encoding="utf-8")

print("Open data/calibration_review.md (Ctrl+Shift+V) and fill in data/human_scores.json")

Open data/calibration_review.md (Ctrl+Shift+V) and fill in data/human_scores.json


In [5]:
from pydantic import BaseModel, Field, model_validator


class HumanScore(BaseModel):
    cal_id: str
    relevance: int = Field(ge=1, le=5)
    specificity: int = Field(ge=1, le=5)
    honesty: int = Field(ge=1, le=5)
    overall: int = Field(ge=1, le=5)
    note: str

    @model_validator(mode="after")
    def check_cap(self):
        if self.honesty <= 2 and self.overall > 2:
            raise ValueError("overall cannot be above 2 when honesty is 2 or below")
        return self


human = [HumanScore.model_validate(row) for row in json.loads(HUMAN_PATH.read_text(encoding="utf-8"))]
print(f"All {len(human)} scores valid")

All 10 scores valid
